# Structural VOI Validation Notebook

This compact notebook exercises the structural EVPI and EVPPI contract with
realistic competing model structures and deterministic sample generation.

In [1]:
import numpy as np

from voiage.methods.structural import structural_evpi, structural_evppi
from voiage.schema import ParameterSet, ValueArray

rng = np.random.default_rng(11)
n_samples = 500

psa_markov = ParameterSet.from_numpy_or_dict({
    "baseline_risk": rng.uniform(0.10, 0.22, n_samples),
    "treatment_effect": rng.normal(0.15, 0.03, n_samples),
})
psa_survival = ParameterSet.from_numpy_or_dict({
    "baseline_risk": rng.uniform(0.08, 0.20, n_samples),
    "treatment_effect": rng.normal(0.18, 0.04, n_samples),
})
psa_des = ParameterSet.from_numpy_or_dict({
    "baseline_risk": rng.uniform(0.09, 0.21, n_samples),
    "treatment_effect": rng.normal(0.17, 0.05, n_samples),
})

evaluators = []

def markov_evaluator(psa: ParameterSet) -> ValueArray:
    baseline = psa.parameters["baseline_risk"]
    effect = psa.parameters["treatment_effect"]
    noise = np.random.default_rng(101).normal(0, 120, size=(psa.n_samples, 2))
    values = np.column_stack(
        [
            18000 + 2200 * baseline - 400 * effect,
            18650 + 2600 * baseline - 520 * effect,
        ]
    ) + noise
    return ValueArray.from_numpy(values, ["Standard care", "Intervention"])

def survival_evaluator(psa: ParameterSet) -> ValueArray:
    baseline = psa.parameters["baseline_risk"]
    effect = psa.parameters["treatment_effect"]
    noise = np.random.default_rng(202).normal(0, 150, size=(psa.n_samples, 2))
    values = np.column_stack(
        [
            17750 + 2400 * baseline - 450 * effect,
            18950 + 3000 * baseline - 580 * effect,
        ]
    ) + noise
    return ValueArray.from_numpy(values, ["Standard care", "Intervention"])

def des_evaluator(psa: ParameterSet) -> ValueArray:
    baseline = psa.parameters["baseline_risk"]
    effect = psa.parameters["treatment_effect"]
    noise = np.random.default_rng(303).normal(0, 180, size=(psa.n_samples, 2))
    values = np.column_stack(
        [
            18100 + 2100 * baseline - 430 * effect,
            18750 + 2850 * baseline - 540 * effect,
        ]
    ) + noise
    return ValueArray.from_numpy(values, ["Standard care", "Intervention"])

evaluators = [markov_evaluator, survival_evaluator, des_evaluator]
probabilities = [0.45, 0.30, 0.25]
psa_samples = [psa_markov, psa_survival, psa_des]


In [2]:
sevpi = structural_evpi(evaluators, probabilities, psa_samples)
sevppi = structural_evppi(evaluators, probabilities, psa_samples, [0, 1])
sevpi_pop = structural_evpi(
    evaluators,
    probabilities,
    psa_samples,
    population=100000,
    time_horizon=10,
    discount_rate=0.03,
)

round(sevpi, 6), round(sevppi, 6), round(sevpi_pop, 6)


(0.049634, 0.0, 42338.587772)

In [3]:
assert sevpi >= 0
assert sevppi >= 0
assert sevppi <= sevpi + 1e-9
assert sevpi_pop >= sevpi
print(f"Structural EVPI: {sevpi:.6f}")
print(f"Structural EVPPI: {sevppi:.6f}")


Structural EVPI: 0.049634
Structural EVPPI: 0.000000
